# B0-2 Learnability Pilot
Tesla T4, Internet, existing GITHUB_TOKEN. One Save & Run All: fail-closed data/init preflight, three fresh512-update arms, exact Pilot-only resume, full probability verification and Release read-back. No Overfit rerun, Test,12M, three-seed main experiment or M6. At most3 hours per session; attach previous Output to resume.

In [ ]:
from pathlib import Path
import os,signal,subprocess,sys,time
started=time.monotonic()
os.environ['CUBLAS_WORKSPACE_CONFIG']=':4096:8'
os.environ['OMP_NUM_THREADS']='1'
os.environ['PYTEST_DISABLE_PLUGIN_AUTOLOAD']='1'
repo=Path('/kaggle/working/minicells-b0-pilot')
branch='research/epclm-004-b0-learnability-pilot'
def run(args,limit=300):
    remaining=min(limit,started+10500-time.monotonic())
    if remaining<=0:raise TimeoutError('Session budget exhausted; preserve Output')
    process=subprocess.Popen(args,cwd=repo if repo.exists() else None,start_new_session=True)
    try:code=process.wait(timeout=remaining)
    except subprocess.TimeoutExpired:
        os.killpg(process.pid,signal.SIGTERM)
        try:process.wait(timeout=15)
        except subprocess.TimeoutExpired:os.killpg(process.pid,signal.SIGKILL);process.wait()
        raise TimeoutError('Pilot stopped at session boundary; preserve and attach Output')
    if code:raise RuntimeError('Pilot stopped; inspect retained Output; do not change frozen data or budget')
if not repo.exists():run(['git','clone','--depth=1','--branch',branch,'https://github.com/ArcheLabs/mini-cells.git',str(repo)])
else:
    origin=subprocess.check_output(['git','remote','get-url','origin'],cwd=repo,text=True).strip()
    if origin.removesuffix('.git')!='https://github.com/ArcheLabs/mini-cells':raise ValueError('Unrelated checkout')
    run(['git','fetch','--depth=1','origin',branch]);run(['git','switch','--detach','FETCH_HEAD'])
base=repo/'research/projects/cellular-language-models/experiments/epclm-004-unified-memory-10m/m5-sv/b0/pilot'
run([sys.executable,'-m','pip','install','-q','einops==0.8.2'])
source=subprocess.check_output(['git','rev-parse','HEAD'],cwd=repo,text=True).strip()
out=Path('/kaggle/working/b0-pilot-'+source[:12]);out.mkdir(exist_ok=True)
# Tiny CPU fixtures only. No canonical Smoke or Overfit is repeated.
proof=out/'cpu-preflight-passed.json'
if not proof.exists():
    run([sys.executable,'-m','pytest','-q',str(base/'tests')],limit=300)
    proof.write_text(__import__('json').dumps({'source_commit':source,'status':'PASS'}))
# This entry audits data and initialization before any outer update.
run([sys.executable,str(base/'run.py'),'--output',str(out),'--execute'],limit=10200)
